"Data Acquisition is already cleard in 100 days of ML, So let's the second step Data Preprocessing."

In [2]:
import numpy as np 
import pandas as pd 

In [ ]:
"""

Text Preparation
 ├── Cleaning
 │    ├── HTML / Tag cleaning
 │    ├── Emoji
 │    └── Spelling checker
 │
 ├── Basic preprocessing
 │    ├── Basic
 │    │    └── Tokenization
 │    │         ├── Sentence
 │    │         └── Word
 │    │
 │    └── Optimal
 │         ├── Stop word removal
 │         ├── Stemming
 │         ├── Removing digits, punctuations
 │         ├── Lowercasing
 │         └── Language detection
 │
 └── Advance preprocessing
      ├── POS tagging
      ├── Parsing
      └── Coreference resolution
"""

# Loading Dataset:

In [11]:
df = pd.read_csv(r"C:\Users\ibad\Desktop\NLP_Notes\Data\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [12]:
df.shape

(50000, 2)

# __1: LowerCasing__:

In [13]:
# Let's take index 3 review

df["review"][3].lower()

"basically there's a family where a little boy (jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />this movie is slower than a soap opera... and suddenly, jake decides to become rambo and kill the zombie.<br /><br />ok, first of all when you're going to make a film you must decide if its a thriller or a drama! as a drama the movie is watchable. parents are divorcing & arguing like in real life. and then we have jake with his closet which totally ruins all the film! i expected to see a boogeyman similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. as for the shots with jake: just ignore them."

In [14]:
# if you want to convert the overall dataset to lowercase

df["review"] = df["review"].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


# __2 Removing HTML Tags:__

In [15]:
import re

text = "<p>This is a <b>good</b> movie.</p>"
clean_text = re.sub(r"<.*?>","",text)
print(clean_text)

"""
r"<.*?>"
< → matches the opening <
.*? → matches anything between < and >
> → matches the closing >
"""
# re.sub() is used to replace matching text.

This is a good movie.


'\nr"<.*?>"\n< → matches the opening <\n.*? → matches anything between < and >\n> → matches the closing >\n'

In [17]:
# You can also create function

import re

def clean_text(text):
    clean = re.sub(r"<.*?>", "", text)
    return clean

clean_text("<p>This is a <b>good</b> movie.</p>")

'This is a good movie.'

In [18]:
# Let's apply it on the data

df['review'] = df['review'].apply(clean_text)
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
49995,i thought this movie did a down right good job...,positive
49996,"bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,i am a catholic taught in parochial elementary...,negative
49998,i'm going to have to disagree with the previou...,negative


# __3 Remove Url:__

In [21]:
import re

def clean_url(text):
    clean = re.sub(r"https?://\S+", "", text)
    return clean

# https? → matches http or https
# :// → matches ://
# \S+ → matches one or more non-space characters

# https?://     → Find the beginning of a URL
# \S+           → Keep matching until whitespace

In [22]:
print(clean_url("Visit https://example.com my website"))
print(clean_url("Vist http://example.com my gethub account"))
print(clean_url("Vist https://www.example.com/page my gethub repository"))

Visit  my website
Vist  my gethub account
Vist  my gethub repository


# __4 Remove Punctuation:__

In [24]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
exclude = string.punctuation

def remove_punctuation(text):
    for char in exclude:
        text = text.replace(char, '')
    return text
text = "Hello, world! This is an amazing movie. I loved it!!! Would you recommend it? Yes, definitely."
print(remove_punctuation(text))

# but this method is very slow

In [29]:
# 2nd Method:

def remove_punc(text):
    return text.translate(str.maketrans('','',exclude))

text = "Hello, world! This is an amazing movie. I loved it!!! Would you recommend it? Yes, definitely."
print(remove_punc(text))

Hello world This is an amazing movie I loved it Would you recommend it Yes definitely


In [33]:
# Applying this on dataset

x_df = pd.read_csv(r'C:\Users\ibad\Desktop\NLP_Notes\Data\train.csv')
x_df.sample(5)

,id,label,tweet
29762,29763,0,"@user if you want to be , be. l tolstoy #ins..."
29858,29859,0,@user why no #atifaslam in #cokestudio this s...
8248,8249,0,@user enjoy your #weekend âð» #friday #...
190,191,0,time to eat with my bae swalscha ðâ¨ð #...
14316,14317,0,"you waiting for something to drop, and you can..."


In [45]:
# Removing punctuation from dataset

x_df['tweet'] = x_df['tweet'].apply(remove_punc)
x_df.sample(5)

,id,label,tweet
19527,19528,0,user user user user user still blaming harper ...
22540,22541,0,user user tongueouttuesday
8849,8850,0,vm meet up in 2 days 20 of us and counting wel...
10966,10967,0,checkout todays trending gif of the day mom ...
20585,20586,0,aww i was on tube and read about this service ...


# __5 Chat Word Treatment:__
it means handling informal words and expressions commonly used in chats or social media.

"u"       → "you"

"r"       → "are"

"gr8"     → "great"

"lol"     → "laughing out loud"

"gonna"   → "going to"

"don't"   → "do not"

In [46]:
chat_word = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ASL": "Age, Sex, Location",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can’t Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don’t Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "PITA": "Pain In The A**",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn’t Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTF": "What The F**k",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired"
}

In [47]:
def chat_converstion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_word:
            new_text.append(chat_word[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)

In [48]:
print(chat_converstion("IMHO he is the best"))

In My Honest/Humble Opinion he is the best


# __6 Spelling Correction:__

In [1]:
from textblob import TextBlob

In [58]:
incorrect_text = "I hav a gud frnd who livs in a beutiful citty. We go to skool together and play footbal evry day."
textBlb = TextBlob(incorrect_text)
print(textBlb.correct().string)

I had a god find who lips in a beautiful city. He go to stool together and play football very day.


# __7 Removing Stop Words:__

Removing common words from text that usually carry little useful information for a particular NLP task.

Common English stop words include:

the, is, a, an, in, on, of, and, to, for, it

In [7]:
from nltk.corpus import stopwords

In [8]:
# To see stop words in english

stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [9]:
def remove_stopwords(text):
    new_text = []

    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [10]:
text = "I am learning natural language processing because it is a very interesting field of artificial intelligence and I want to use it for my future projects."
remove_stopwords(text)

'I  learning natural language processing      interesting field  artificial intelligence  I want  use    future projects.'

In [21]:
# if you have to apply on all the dataset

df['review'] = df['review'].apply(remove_stopwords)
df.sample(5)

,review,sentiment
35558,well probably agree bad comments movie c...,negative
13788,saw cartoon first time recognized caric...,positive
41962,man wrongfully accused killing friend ai...,positive
7736,"solid, unremarkable film. matthau, einstein...",positive
42428,watch show every day entertaining. prov...,positive


# __8 Handling Emojies:__

In [ ]:
# If there is emojies in your data there are three option, 1: Remove it, 2: replace, 3: Keep it

# 1: Remove
import re

def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Symbols & pictographs
        "\U0001F680-\U0001F6FF"  # Transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # Flags
        "\U0001F900-\U0001F9FF"  # Supplemental symbols & pictographs
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', text)

text = "I am happy 🥰🥰"

print(remove_emojis(text))

I am happy 


In [ ]:
# 2 Convert 

import emoji

text = "I love this movie 😍"

print(emoji.demojize(text))
